# BÁO CÁO GIAI ĐOẠN 2: TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING PIPELINE)
## Đề tài: Xây dựng bộ dữ liệu chuyến bay chậm, hủy và đúng giờ của các hãng hàng không Việt Nam
**Môn học:** Khai thác dữ liệu (Data Mining)  
**Mục tiêu Notebook:**
1. Thực thi luồng chuẩn hóa: Data Cleaning -> Integration -> Transformation -> Scaling -> Encoding.
2. Biến đổi không gian dữ liệu đầu vào phục vụ mô hình hóa và khai phá luật kết hợp.
3. Ngăn ngừa triệt để rò rỉ dữ liệu (Data Leakage) và rò rỉ phân tách thời gian (Temporal Leakage).
4. Đóng gói bộ dữ liệu sạch và các thành phần tiền xử lý (Scaler, Encoder, Metadata) để tái sử dụng.

In [1]:
# 1. THIẾT LẬP MÔI TRƯỜNG VÀ KHAI BÁO THƯ VIỆN
import os
import glob
import json
import re
import unicodedata
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
import joblib

# Thiết lập đường dẫn thư mục
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
WEATHER_FILE = BASE_DIR / "data" / "weather_hourly.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Môi trường đã sẵn sàng! Thư mục xuất dữ liệu:", PROCESSED_DIR)

NameError: name 'Path' is not defined

---
## BƯỚC 1: TIẾP NHẬN VÀ LÀM SẠCH DỮ LIỆU THÔ (DATA CLEANING)

In [ ]:
# 1.1. Nạp toàn bộ dữ liệu thô từ thư mục data/raw
raw_files = sorted(glob.glob(str(RAW_DIR / "*.csv")))
df_raw = pd.concat([pd.read_csv(f, dtype=str, keep_default_na=False) for f in raw_files], ignore_index=True)
initial_count = len(df_raw)
print(f"Tổng số bản ghi thô nạp vào: {initial_count:,}")

# 1.2. Chuẩn hóa chuỗi và Lọc theo phạm vi nghiên cứu (Hãng hàng không Việt Nam)
VN_AIRLINES = {'VN', 'VJ', 'QH', 'BL', 'VU', '9G', '0V'}

for c in ['airport', 'direction', 'flight_date', 'flight_no', 'scheduled_time', 'status', 'airline_code']:
    if c in df_raw.columns:
        df_raw[c] = df_raw[c].astype(str).str.strip().str.upper()

# Chuẩn hóa số hiệu chuyến bay (bỏ số 0 thừa)
df_raw['flight_no'] = df_raw['flight_no'].str.replace(r'\s+', '', regex=True)
df_raw['flight_no'] = df_raw['flight_no'].str.replace(r'^([A-Z0-9]{2})0+(\d)', r'\1\2', regex=True)

# Lọc phạm vi đề tài
df_cleaned = df_raw[df_raw['airline_code'].isin(VN_AIRLINES)].copy()
dropped_scope = initial_count - len(df_cleaned)
print(f"Đã lọc {dropped_scope:,} bản ghi thuộc hãng quốc tế/cargo. Còn lại: {len(df_cleaned):,} bản ghi hãng VN.")

In [ ]:
# 1.3. Tính toán số phút trễ (Hỗ trợ giải quyết giờ qua nửa đêm)
def parse_time_min(t_str):
    if not t_str or pd.isna(t_str): return None, 0
    m = re.search(r'(\d{1,2}):(\d{2})\s*([+-]?)', str(t_str))
    if not m: return None, 0
    h, mins, sign = int(m.group(1)), int(m.group(2)), m.group(3)
    return h * 60 + mins, {'-': -1, '+': 1, '': 0}[sign]

def calculate_delay(sched, est, actual, status_str):
    m = re.search(r'(\d+)\s*minutes?\s*late', str(status_str), re.IGNORECASE)
    if m: return float(m.group(1))
    target = actual if (actual and str(actual).strip()) else est
    s_min, _ = parse_time_min(sched)
    t_min, off = parse_time_min(target)
    if s_min is None or t_min is None: return np.nan
    diff = (t_min + off * 1440) - s_min
    if off == 0:
        if diff < -720: diff += 1440
        elif diff > 720: diff -= 1440
    return float(diff)

df_cleaned['delay_min'] = df_cleaned.apply(
    lambda r: calculate_delay(r['scheduled_time'], r.get('estimated_time', ''),
                              r.get('actual_time', ''), r.get('status', '')), axis=1
)

# 1.4. Khử trùng lặp đa nguồn (Deduplication)
key_event = ['airport', 'direction', 'flight_date', 'flight_no', 'scheduled_time']
df_cleaned['has_info'] = (~df_cleaned['status'].isin(['', '(TRỐNG)', 'SCHEDULED'])).astype(int)
df_cleaned = df_cleaned.sort_values(by=key_event + ['has_info', 'scraped_at'])
df_dedup = df_cleaned.drop_duplicates(subset=key_event, keep='last').copy()
print(f"Số sự kiện duy nhất sau khi khử trùng đa nguồn: {len(df_dedup):,}")

---
## BƯỚC 2: TÍCH HỢP DỮ LIỆU ĐA NGUỒN (DATA INTEGRATION)

In [ ]:
# 2.1. Ánh xạ địa danh sân bay đối ứng sang mã IATA
CITY_IATA = {
    'hochiminh': 'SGN', 'saigon': 'SGN', 'tansonnhat': 'SGN',
    'hanoi': 'HAN', 'noibai': 'HAN', 'danang': 'DAD',
    'nhatrang': 'CXR', 'camranh': 'CXR', 'phuquoc': 'PQC',
    'haiphong': 'HPH', 'catbi': 'HPH', 'vinh': 'VII', 'hue': 'HUI',
    'quynhon': 'UIH', 'cantho': 'VCA', 'dalat': 'DLI', 'buonmathuot': 'BMV',
    'vandon': 'VDO', 'thanhhoa': 'THD', 'chulai': 'VCL', 'pleiku': 'PXU'
}

def to_iata(s):
    u = str(s or '').upper()
    m = re.search(r'\(([A-Z]{3})\)', u)
    if m: return m.group(1)
    if re.fullmatch(r'\s*[A-Z]{3}\s*', u): return u.strip()
    clean_s = re.sub(r'[^a-z0-9]+', '', unicodedata.normalize('NFD', str(s or '')).encode('ascii', 'ignore').decode('utf-8').lower())
    for k, v in CITY_IATA.items():
        if k in clean_s: return v
    return 'OTHER'

df_dedup['other_airport_code'] = df_dedup['other_airport'].apply(to_iata)

# 2.2. Tích hợp dữ liệu thời tiết (Open-Meteo Hourly)
df_dedup['scheduled_hour'] = df_dedup['scheduled_time'].str.extract(r'^(\d{1,2}):')[0].astype(float).fillna(0).astype(int)

if WEATHER_FILE.exists():
    df_wx = pd.read_csv(WEATHER_FILE)
    df_wx['flight_date'] = df_wx['time'].str[:10]
    df_wx['scheduled_hour'] = df_wx['time'].str[11:13].astype(int)
    
    # Left join thời tiết theo sân bay, ngày và giờ
    df_integrated = df_dedup.merge(
        df_wx[['airport', 'flight_date', 'scheduled_hour', 'precipitation', 'wind_gusts_10m', 'visibility', 'weather_code']],
        on=['airport', 'flight_date', 'scheduled_hour'],
        how='left'
    )
    print("Tích hợp thời tiết thành công!")
else:
    print("Cảnh báo: Không tìm thấy weather_hourly.csv! Tạo đặc trưng thời tiết mặc định.")
    df_integrated = df_dedup.copy()
    for col in ['precipitation', 'wind_gusts_10m', 'visibility', 'weather_code']:
        df_integrated[col] = 0.0

print(f"Kích thước bảng dữ liệu sau tích hợp: {df_integrated.shape}")

---
## BƯỚC 3: BIẾN ĐỔI ĐẶC TRƯNG VÀ THIẾT LẬP NHÃN (TRANSFORMATION)

In [ ]:
# 3.1. Thiết lập nhãn mục tiêu Ground Truth (on_time, delayed, cancelled)
def label_ground_truth(row, delay_threshold=15):
    st_raw = str(row.get('status', '')).strip().lower()
    dly = row.get('delay_min', np.nan)
    if re.search(r'cancel|huy|hủy', st_raw): return 'cancelled'
    if not st_raw or st_raw in {'scheduled', '(trống)'}: return 'unknown'
    is_done = bool(re.search(r'arrive|depart|land|hacan|catcan|on-time|ontime', st_raw))
    if pd.notna(dly):
        if dly >= delay_threshold: return 'delayed'
        elif dly >= -60: return 'on_time'
        return 'unknown'
    if is_done:
        return 'delayed' if 'late' in st_raw else 'on_time'
    return 'unknown'

df_integrated['target_status'] = df_integrated.apply(label_ground_truth, axis=1)
# Loại bỏ các chuyến bay chưa chốt trạng thái (unknown)
df_modeled = df_integrated[df_integrated['target_status'].isin(['on_time', 'delayed', 'cancelled'])].copy()

# 3.2. Biến đổi đặc trưng chu kỳ thời gian (Sin/Cos Encoding)
h = df_modeled['scheduled_hour']
df_modeled['hour_sin'] = np.sin(2 * np.pi * h / 24.0)
df_modeled['hour_cos'] = np.cos(2 * np.pi * h / 24.0)

# Khung giờ khai thác
def assign_slot(hr):
    return '00h-06h' if hr < 6 else '06h-12h' if hr < 12 else '12h-18h' if hr < 18 else '18h-24h'
df_modeled['time_slot'] = df_modeled['scheduled_hour'].apply(assign_slot)

# Ngày trong tuần
dt = pd.to_datetime(df_modeled['flight_date'], errors='coerce')
dow = dt.dt.dayofweek
df_modeled['dow_sin'] = np.sin(2 * np.pi * dow / 7.0)
df_modeled['dow_cos'] = np.cos(2 * np.pi * dow / 7.0)
df_modeled['is_weekend'] = (dow >= 5).astype(int)

# 3.3. Đặc trưng tải trọng sân bay và Phân loại tuyến bay
df_modeled['airport_load'] = df_modeled.groupby(['airport', 'flight_date', 'scheduled_hour'])['flight_no'].transform('count')

VN_AIRPORTS = {'HAN', 'SGN', 'DAD', 'CXR', 'PQC', 'HPH', 'VII', 'HUI', 'UIH', 'VCA', 'DLI', 'BMV', 'VDO', 'THD', 'VCL', 'PXU'}
df_modeled['route_type'] = df_modeled.apply(
    lambda r: 'Domestic' if (r['airport'] in VN_AIRPORTS and r['other_airport_code'] in VN_AIRPORTS) else 'International',
    axis=1
)

# Đặc trưng thời tiết nhị phân
df_modeled['precipitation'] = pd.to_numeric(df_modeled['precipitation'], errors='coerce').fillna(0)
df_modeled['wind_gusts_10m'] = pd.to_numeric(df_modeled['wind_gusts_10m'], errors='coerce').fillna(0)
df_modeled['is_bad_weather'] = ((df_modeled['precipitation'] >= 2.0) | (df_modeled['wind_gusts_10m'] >= 35.0)).astype(int)

print("Phân phối nhãn mục tiêu:")
print(df_modeled['target_status'].value_counts())

---
## BƯỚC 4 & 5: SCALING, ENCODING VÀ CHIA TẬP KHÔNG RÒ RỈ DỮ LIỆU

In [ ]:
# 4.1. Phân tách tập Train / Test theo chuỗi thời gian (Temporal Split 80/20)
df_modeled = df_modeled.sort_values(by=['flight_date', 'scheduled_time']).reset_index(drop=True)
split_idx = int(len(df_modeled) * 0.8)
train_df = df_modeled.iloc[:split_idx].copy()
test_df = df_modeled.iloc[split_idx:].copy()

print(f"Tập huấn luyện (Train): {len(train_df):,} dòng | Tập kiểm thử (Test): {len(test_df):,} dòng")

# 4.2. Chuẩn hóa biến số (RobustScaler fit trên Train)
numeric_cols = ['airport_load', 'precipitation', 'wind_gusts_10m', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
scaler = RobustScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

# 4.3. Mã hóa biến phân loại (One-Hot Encoder fit trên Train)
cat_cols = ['airline_code', 'airport', 'direction', 'route_type', 'time_slot']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(train_df[cat_cols])

encoded_feature_names = ohe.get_feature_names_out(cat_cols)
train_ohe = pd.DataFrame(ohe.transform(train_df[cat_cols]), columns=encoded_feature_names, index=train_df.index)
test_ohe = pd.DataFrame(ohe.transform(test_df[cat_cols]), columns=encoded_feature_names, index=test_df.index)

# 4.4. Mã hóa nhãn mục tiêu
label_map = {'on_time': 0, 'delayed': 1, 'cancelled': 2}
train_df['target'] = train_df['target_status'].map(label_map)
test_df['target'] = test_df['target_status'].map(label_map)

# Kết hợp thành tập đặc trưng hoàn chỉnh X và nhãn y (Ngăn ngừa rò rỉ dữ liệu)
X_train = pd.concat([train_df[numeric_cols + ['is_weekend', 'is_bad_weather']], train_ohe], axis=1)
y_train = train_df['target']
X_test = pd.concat([test_df[numeric_cols + ['is_weekend', 'is_bad_weather']], test_ohe], axis=1)
y_test = test_df['target']

print(f"Kích thước không gian đặc trưng đầu vào X_train: {X_train.shape}")
print(f"Kích thước không gian đặc trưng đầu vào X_test:  {X_test.shape}")

---
## BƯỚC 6: XUẤT DỮ LIỆU ĐÃ TIỀN XỬ LÝ VÀ ĐÓNG GÓI ARTIFACTS

In [ ]:
# 6.1. Lưu bộ dữ liệu tiền xử lý (CSV và Parquet)
processed_full = pd.concat([
    pd.concat([X_train, y_train, train_df[['flight_no', 'flight_date', 'scheduled_time', 'target_status']]], axis=1),
    pd.concat([X_test, y_test, test_df[['flight_no', 'flight_date', 'scheduled_time', 'target_status']]], axis=1)
], ignore_index=True)

csv_out = PROCESSED_DIR / "flights_processed.csv"
parquet_out = PROCESSED_DIR / "flights_processed.parquet"
processed_full.to_csv(csv_out, index=False, encoding='utf-8-sig')
try:
    processed_full.to_parquet(parquet_out, index=False)
    print(f"-> Đã lưu Parquet: {parquet_out}")
except Exception as e:
    print("Lưu ý: Chưa có thư viện pyarrow/fastparquet. Bỏ qua xuất Parquet.")

# 6.2. Lưu Scaler, Encoder và Metadata tái sử dụng
joblib.dump(scaler, PROCESSED_DIR / "scaler.joblib")
joblib.dump(ohe, PROCESSED_DIR / "encoder.joblib")

meta = {
    'numeric_columns': numeric_cols,
    'categorical_columns': cat_cols,
    'one_hot_columns': list(encoded_feature_names),
    'label_mapping': label_map,
    'created_at': datetime.now().isoformat(),
    'train_samples': len(X_train),
    'test_samples': len(X_test)
}
with open(PROCESSED_DIR / "preprocessing_metadata.json", 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"-> Đã lưu CSV: {csv_out}")
print(f"-> Đã lưu Metadata & Scalers vào thư mục: {PROCESSED_DIR}")
print("HOÀN THÀNH TOÀN BỘ QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU!")